In [1]:
import pandas as pd
import numpy as np
import os
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

In [2]:
# Constants
SYNTHETIC_SAMPLES = 10000
DATA_PATH = "../../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
METADATA_FILE = "spoofing_metadata.json"

In [3]:
# Load the dataset
try:
    df = pd.read_excel(DATA_PATH)
except FileNotFoundError:
    print(f"Error: The file '{DATA_PATH}' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [4]:
# Load or create metadata
if os.path.exists(METADATA_FILE):
    metadata = Metadata.load_from_json(METADATA_FILE)
    print(f"Loaded metadata from {METADATA_FILE}")
else:
    metadata = Metadata()
    metadata.detect_table_from_dataframe(
        data=df,
        table_name='spoofing_data'
    )
    metadata.save_to_json(METADATA_FILE)
    print(f"Created and saved new metadata to {METADATA_FILE}")

Loaded metadata from spoofing_metadata.json


In [7]:
# Synthesizer configurations
# If you don't have a GPU available, it will automatically use CPU.
synthesizers = [
    {
        # Uses GAN-based, deep learning methods to train a model and generate synthetic data.
        'name': 'CTGAN',
        'class': CTGANSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'verbose': True,
            'cuda': True
        },
        'output': "../../../../Datasets/excel/2dSpoofingDataset/spoofing-CTGAN-synthetic.csv"
    },
    {
        # Uses classic, statistical methods to train a model and generate synthetic data
        'name': 'GaussianCopula',
        'class': GaussianCopulaSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
        },
        'output': "../../../../Datasets/excel/2dSpoofingDataset/spoofing-GaussianCopula-synthetic.csv"
    },
    {
        # Uses a variational autoencoder (VAE)-based, neural network techniques to train a model and generate synthetic data.
        'name': 'TVAE',
        'class': TVAESynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'verbose': True,
            'cuda': True
        },
        'output': "../../../../Datasets/excel/2dSpoofingDataset/spoofing-TVAE-synthetic.csv"
    }
]

In [8]:
# Fit synthesizers and generate synthetic data
# This will take some time.
for synth in synthesizers:
    print(f"\nProcessing {synth['name']}...")
    SynthClass = synth['class']
    if synth['name'] == 'GaussianCopula':
        synthesizer = SynthClass(metadata)
    else:
        synthesizer = SynthClass(metadata, **synth['params'])
    synthesizer.fit(df)
    synthetic_data = synthesizer.sample(num_rows=SYNTHETIC_SAMPLES)
    print(f"Synthetic data preview for {synth['name']}:")
    print(synthetic_data.head())
    synthetic_data.to_csv(synth['output'], index=False)
    print(f"Synthetic dataset saved to {synth['output']}")


Processing CTGAN...


Gen. (-0.42) | Discrim. (0.08): 100%|██████████| 300/300 [3:50:04<00:00, 46.02s/it]   


Synthetic data preview for CTGAN:
   PRN           DO            PD             RX            TOW  \
0    8  4439.392679  4.305831e+06  263422.230044  263480.647534   
1    8  4437.082057  4.424044e+06  263511.713180  263541.914517   
2   11  3249.174829  3.162239e+05  491632.520544  491628.614669   
3    2  2460.129547  2.559038e+07  492039.420000  492039.341092   
4    4  -570.312146  2.728845e+06  263301.352577  263281.353817   

              CP             EC             LC             PC            PIP  \
0 -835186.099880   72158.906823   76006.959605   95462.971072  -92127.294083   
1 -236058.084072   74161.287340   82053.509877   76379.064859  -58880.028344   
2  -89443.968189  142930.372911  151563.121693  178334.809081  102173.852718   
3 -533536.643422  138841.880582  109798.098463  142858.435278 -177031.917511   
4  140953.149743   92379.061041  122971.794302  121551.679352  116916.895012   

            PQP          TCD        CN0  Output  
0  -2197.940327  4529.912979  38

Loss: -43.974:   7%|▋         | 140/2000 [44:02<9:45:06, 18.87s/it] 


KeyboardInterrupt: 